In [4]:
import pandas as pd
import glob
all_data = []
files = glob.glob(r"C:\Users\Bathini pavithra\Downloads\all-india-villages-master-list-excel\dataset\*.xls")
print("Files found:", files)
for file in files:
    print("\nProcessing:", file)
    df = pd.read_excel(file, engine='xlrd')
    #  Print columns for debugging
    print("Original Columns:", df.columns)
    #  Normalize column names (lowercase + strip spaces)
    df.columns = df.columns.str.strip().str.lower()
    print("Cleaned Columns:", df.columns)
    # Rename dynamically
    df = df.rename(columns={
        'state name': 'state',
        'district name': 'district',
        'sub-district name': 'sub_district',
        'area name': 'village'
    })
    # Check if required columns exist
    required_cols = ['state', 'district', 'sub_district', 'village']
    if not all(col in df.columns for col in required_cols):
        print(" Skipping file due to missing columns:", file)
        continue
    df = df[required_cols]
    # Clean data
    df = df.dropna()
    df = df.drop_duplicates()
    for col in df.columns:
        df[col] = df[col].astype(str).str.strip()
    all_data.append(df)
#  Final check
if len(all_data) == 0:
    print("❌ No valid data processed")
else:
    final_df = pd.concat(all_data)
    final_df.to_csv("cleaned_villages.csv", index=False)
    print("\n✅ Data cleaned successfully!")
    print("Total rows:", len(final_df))

Files found: ['C:\\Users\\Bathini pavithra\\Downloads\\all-india-villages-master-list-excel\\dataset\\Rdir_2011_02_HIMACHAL_PRADESH.xls', 'C:\\Users\\Bathini pavithra\\Downloads\\all-india-villages-master-list-excel\\dataset\\Rdir_2011_03_PUNJAB.xls', 'C:\\Users\\Bathini pavithra\\Downloads\\all-india-villages-master-list-excel\\dataset\\Rdir_2011_06_HARYANA.xls', 'C:\\Users\\Bathini pavithra\\Downloads\\all-india-villages-master-list-excel\\dataset\\Rdir_2011_08_RAJASTHAN.xls', 'C:\\Users\\Bathini pavithra\\Downloads\\all-india-villages-master-list-excel\\dataset\\Rdir_2011_10_BIHAR.xls', 'C:\\Users\\Bathini pavithra\\Downloads\\all-india-villages-master-list-excel\\dataset\\Rdir_2011_11_SIKKIM.xls', 'C:\\Users\\Bathini pavithra\\Downloads\\all-india-villages-master-list-excel\\dataset\\Rdir_2011_12_ARUNACHAL_PRADESH.xls', 'C:\\Users\\Bathini pavithra\\Downloads\\all-india-villages-master-list-excel\\dataset\\Rdir_2011_13_NAGALAND.xls', 'C:\\Users\\Bathini pavithra\\Downloads\\all-ind

In [11]:
import pandas as pd
import psycopg2

# Load data
df = pd.read_csv("cleaned_villages.csv")

# Connect PostgreSQL
conn = psycopg2.connect(
    host="localhost",
    database="villages_db",
    user="postgres",
    password="postgres123"
)

cur = conn.cursor()

state_map = {}
district_map = {}
sub_district_map = {}

for index, row in df.iterrows():

    # -------- STATE --------
    state = row['state']
    if state not in state_map:
        cur.execute("""
            INSERT INTO states (name)
            VALUES (%s)
            ON CONFLICT (name) DO NOTHING
            RETURNING id
        """, (state,))
        
        result = cur.fetchone()
        if result:
            state_map[state] = result[0]
        else:
            cur.execute("SELECT id FROM states WHERE name=%s", (state,))
            state_map[state] = cur.fetchone()[0]

    state_id = state_map[state]

    # -------- DISTRICT --------
    district_key = (row['district'], state_id)
    if district_key not in district_map:
        cur.execute("""
            INSERT INTO districts (name, state_id)
            VALUES (%s, %s)
            RETURNING id
        """, (row['district'], state_id))
        
        district_map[district_key] = cur.fetchone()[0]

    district_id = district_map[district_key]

    # -------- SUB DISTRICT --------
    sub_key = (row['sub_district'], district_id)
    if sub_key not in sub_district_map:
        cur.execute("""
            INSERT INTO sub_districts (name, district_id)
            VALUES (%s, %s)
            RETURNING id
        """, (row['sub_district'], district_id))
        
        sub_district_map[sub_key] = cur.fetchone()[0]

    sub_district_id = sub_district_map[sub_key]

    # -------- VILLAGE --------
    cur.execute("""
        INSERT INTO villages (name, sub_district_id)
        VALUES (%s, %s)
    """, (row['village'], sub_district_id))

    # 🔄 Commit every 5000 rows (important for performance)
    if index % 5000 == 0:
        conn.commit()
        print(f"Inserted {index} rows...")

# Final commit
conn.commit()

cur.close()
conn.close()

print("✅ ALL DATA INSERTED SUCCESSFULLY!")

Inserted 0 rows...
Inserted 5000 rows...
Inserted 10000 rows...
Inserted 15000 rows...
Inserted 20000 rows...
Inserted 25000 rows...
Inserted 30000 rows...
Inserted 35000 rows...
Inserted 40000 rows...
Inserted 45000 rows...
Inserted 50000 rows...
Inserted 55000 rows...
Inserted 60000 rows...
Inserted 65000 rows...
Inserted 70000 rows...
Inserted 75000 rows...
Inserted 80000 rows...
Inserted 85000 rows...
Inserted 90000 rows...
Inserted 95000 rows...
Inserted 100000 rows...
Inserted 105000 rows...
Inserted 110000 rows...
Inserted 115000 rows...
Inserted 120000 rows...
Inserted 125000 rows...
Inserted 130000 rows...
Inserted 135000 rows...
Inserted 140000 rows...
Inserted 145000 rows...
Inserted 150000 rows...
Inserted 155000 rows...
Inserted 160000 rows...
Inserted 165000 rows...
Inserted 170000 rows...
Inserted 175000 rows...
Inserted 180000 rows...
Inserted 185000 rows...
Inserted 190000 rows...
Inserted 195000 rows...
Inserted 200000 rows...
Inserted 205000 rows...
Inserted 210000 r